# Validação do Forecast de Custos — Painel Executivo

Compara o **custo unitário previsto** (Custo Média Móvel) com o **custo unitário real**, por Centro + Material + Mês (jan–jun/2026), sobre o `df_merged` já pareado.

**Conceito central da medição:** os erros são calculados sobre os preços unitários, mas cada linha pesa na acurácia conforme o **dinheiro que movimentou** (custo real × produção). Errar o preço de um material de alto volume importa mais do que errar o de um material marginal.

**4 indicadores:**

| Indicador | O que responde |
|---|---|
| **Acurácia (1−WMAPE)** | Quão perto o forecast chegou, ponderado pelo valor movimentado |
| **MAE (R$/un)** | Em média, quantos reais por unidade erramos no preço |
| **Viés** | O forecast tende a superestimar ou subestimar os custos? |
| **Hit Rate** | % de previsões dentro da tolerância por classe de material |

## 1. Configuração

In [ ]:
import pandas as pd
import numpy as np

# ---- Nomes das colunas no df_merged (ajuste se necessário) ----
COL_CENTRO   = "Centro"
COL_MATERIAL = "Material"
COL_MES      = "Mês"
COL_PREV     = "Custo Média Móvel"   # custo UNITÁRIO previsto
COL_REAL     = "Custo"               # custo UNITÁRIO real
COL_PROD     = "Produção"            # produção REAL (da base de comparação)
                                     # se o merge gerou sufixo, ex.: "Produção_real", ajuste aqui

# ---- Tolerância de acerto por classe ABC (limite de erro % no preço) ----
TOLERANCIAS = {"A": 0.05, "B": 0.10, "C": 0.20}

# ---- Cortes da curva ABC (participação acumulada no VALOR movimentado) ----
CORTE_A = 0.80
CORTE_B = 0.95

TOP_N_OFENSORES = 15

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", None)

## 2. Preparação e cálculo dos erros

Converte tipos, garante **1 linha por Centro + Material + Mês** (o output do forecast é aberto por Componente e repetiria o material), descarta linhas inválidas para o cálculo, computa erros unitários e o valor movimentado, classifica ABC e aplica a tolerância.

In [ ]:
base = df_merged.copy()

for c in [COL_PREV, COL_REAL, COL_PROD]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# ---- Garantir 1 linha por chave (dedup de linhas abertas por Componente) ----
n_antes = len(base)
base = base.drop_duplicates(subset=[COL_CENTRO, COL_MATERIAL, COL_MES])
n_dedup = n_antes - len(base)

# ---- Linhas inválidas para o cálculo percentual ----
n_total = len(base)
base = base[base[COL_PREV].notna() & base[COL_REAL].notna()
            & (base[COL_REAL] != 0) & base[COL_PROD].notna() & (base[COL_PROD] > 0)]
n_excluidas = n_total - len(base)

# ---- Erros no PREÇO unitário ----
base["Erro_Un_R$"]     = base[COL_PREV] - base[COL_REAL]      # com sinal, R$/unidade
base["Erro_Un_Abs_R$"] = base["Erro_Un_R$"].abs()
base["Erro_%"]         = base["Erro_Un_R$"] / base[COL_REAL]  # com sinal
base["APE"]            = base["Erro_%"].abs()

# ---- Valor movimentado (peso da linha) e erro em dinheiro ----
base["Valor_Real_R$"]     = base[COL_REAL] * base[COL_PROD]
base["Valor_Prev_R$"]     = base[COL_PREV] * base[COL_PROD]
base["Erro_Valor_Abs_R$"] = base["Erro_Un_Abs_R$"] * base[COL_PROD]

# ---- Classe ABC pelo VALOR movimentado acumulado do período ----
valor_mat = (base.groupby([COL_CENTRO, COL_MATERIAL])["Valor_Real_R$"].sum()
             .sort_values(ascending=False).reset_index())
valor_mat["Acum"] = valor_mat["Valor_Real_R$"].cumsum() / valor_mat["Valor_Real_R$"].sum()
valor_mat["Classe_ABC"] = np.select(
    [valor_mat["Acum"] <= CORTE_A, valor_mat["Acum"] <= CORTE_B], ["A", "B"], "C")
base = base.merge(valor_mat[[COL_CENTRO, COL_MATERIAL, "Classe_ABC"]],
                  on=[COL_CENTRO, COL_MATERIAL], how="left")

# ---- Acerto dentro da tolerância da classe ----
base["Acertou"] = base["APE"] <= base["Classe_ABC"].map(TOLERANCIAS)

print(f"Linhas duplicadas removidas (abertura por componente): {n_dedup:,}")
print(f"Linhas analisadas: {len(base):,} (excluídas por dados inválidos: {n_excluidas:,})")

## 3. Painel executivo — os 4 números

In [ ]:
wmape    = base["Erro_Valor_Abs_R$"].sum() / base["Valor_Real_R$"].sum()
acuracia = 1 - wmape
mae_un   = base["Erro_Un_Abs_R$"].mean()
vies     = (base["Valor_Prev_R$"].sum() - base["Valor_Real_R$"].sum()) / base["Valor_Real_R$"].sum()
hit_rate = base["Acertou"].mean()

painel = pd.DataFrame({
    "Indicador": ["Acurácia (1−WMAPE)", "MAE — erro médio no preço unitário",
                  "Viés", "Hit Rate"],
    "Resultado": [f"{acuracia:.1%}", f"R$ {mae_un:,.2f} /un",
                  f"{vies:+.1%}", f"{hit_rate:.1%}"],
    "Leitura": [
        "Ponderado pelo valor movimentado (custo real × produção)",
        "Distância média entre preço previsto e preço real, por material/mês",
        "Positivo = forecast superestima | Negativo = subestima (em R$)",
        "% de previsões dentro da tolerância (A: ±5% | B: ±10% | C: ±20%)",
    ],
})
display(painel.style.hide(axis="index"))

## 4. Acurácia por classe de material

Classe definida pelo **valor movimentado** (custo × produção) acumulado no semestre. A leitura que importa: **como está a classe A**, que concentra ~80% do dinheiro.

In [ ]:
por_classe = (
    base.groupby("Classe_ABC")
    .apply(lambda x: pd.Series({
        "Materiais":   x[COL_MATERIAL].nunique(),
        "Valor_Real":  x["Valor_Real_R$"].sum(),
        "Acuracia":    1 - x["Erro_Valor_Abs_R$"].sum() / x["Valor_Real_R$"].sum(),
        "MAE_Un_R$":   x["Erro_Un_Abs_R$"].mean(),
        "Vies":        (x["Valor_Prev_R$"].sum() - x["Valor_Real_R$"].sum()) / x["Valor_Real_R$"].sum(),
        "Hit_Rate":    x["Acertou"].mean(),
    }), include_groups=False)
    .reset_index()
)
por_classe["%_do_Valor"] = por_classe["Valor_Real"] / por_classe["Valor_Real"].sum()

display(
    por_classe[["Classe_ABC", "Materiais", "%_do_Valor",
                "Acuracia", "MAE_Un_R$", "Vies", "Hit_Rate"]]
    .style.hide(axis="index")
    .format({"%_do_Valor": "{:.1%}", "Acuracia": "{:.1%}",
             "MAE_Un_R$": "R$ {:,.2f}", "Vies": "{:+.1%}", "Hit_Rate": "{:.1%}"})
)

## 5. Onde está o dinheiro do erro

Top materiais por **impacto em R$** (erro no preço × volume produzido) — a lista de prioridade caso seja preciso melhorar o forecast.

In [ ]:
top_ofensores = (
    base.groupby([COL_CENTRO, COL_MATERIAL, "Classe_ABC"])
    .agg(Valor_Real=("Valor_Real_R$", "sum"),
         Erro_Impacto=("Erro_Valor_Abs_R$", "sum"),
         Preco_Real_Medio=(COL_REAL, "mean"),
         Preco_Prev_Medio=(COL_PREV, "mean"),
         APE_Medio=("APE", "mean"))
    .reset_index()
    .rename(columns={"Erro_Impacto": "Erro_Impacto_R$"})
)
top_ofensores["%_do_Erro_Total"] = (
    top_ofensores["Erro_Impacto_R$"] / top_ofensores["Erro_Impacto_R$"].sum()
)

display(
    top_ofensores.sort_values("Erro_Impacto_R$", ascending=False)
    .head(TOP_N_OFENSORES)
    .style.hide(axis="index")
    .format({"Valor_Real": "R$ {:,.2f}", "Erro_Impacto_R$": "R$ {:,.2f}",
             "Preco_Real_Medio": "R$ {:,.4f}", "Preco_Prev_Medio": "R$ {:,.4f}",
             "APE_Medio": "{:.1%}", "%_do_Erro_Total": "{:.1%}"})
)

## 6. Visual único para a apresentação

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Acurácia por classe
cores = {"A": "#2a9d8f", "B": "#e9c46a", "C": "#e76f51"}
axes[0].bar(por_classe["Classe_ABC"], por_classe["Acuracia"],
            color=[cores[c] for c in por_classe["Classe_ABC"]])
axes[0].axhline(acuracia, color="gray", ls="--", lw=1,
                label=f"Geral: {acuracia:.1%}")
axes[0].set_title("Acurácia do Forecast por classe de material")
axes[0].set_ylim(0, 1.05)
axes[0].legend()
for i, v in enumerate(por_classe["Acuracia"]):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")

# Valor total: Real x Previsto (nas quantidades reais)
vr, vp = base["Valor_Real_R$"].sum(), base["Valor_Prev_R$"].sum()
axes[1].bar(["Real", "Previsto"], [vr, vp], color=["#457b9d", "#a8dadc"])
axes[1].set_title(f"Valor movimentado: Real x Previsto (viés {vies:+.1%})")
for i, v in enumerate([vr, vp]):
    axes[1].text(i, v, f"R$ {v:,.0f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 7. Exportação (opcional)

In [ ]:
with pd.ExcelWriter("validacao_forecast_executivo.xlsx") as writer:
    painel.to_excel(writer, sheet_name="Painel", index=False)
    por_classe.to_excel(writer, sheet_name="Por_Classe", index=False)
    top_ofensores.sort_values("Erro_Impacto_R$", ascending=False).to_excel(
        writer, sheet_name="Top_Ofensores", index=False)
    base.to_excel(writer, sheet_name="Base_Detalhada", index=False)

print("Arquivo 'validacao_forecast_executivo.xlsx' gerado.")